# Run streamlined MIMIC experiments

Run or resume an experiment profile and rebuild its derived tables and figures. Set `RESTART = True` only when all saved raw condition results should be discarded and recomputed.


In [ ]:
# The full YAML supplies datasets, seeds, ratios, and training sizes.
DATASET_N_ROWS = 5000
EXPERIMENT_BASE_NAME = "latent-displacement-best"

# False resumes from raw/condition_results.csv and skips completed conditions.
# True deletes those raw results and reruns every condition from scratch.
RESTART = False

# Display the updating text progress bar during training.
SHOW_PROGRESS = True

# Latent/direct sampling hyperparameters for this run.
MIMIC_MODE = "factorised"
CAPACITY = 0.5
N_NEIGHBORS = 14
LAMBDA_RANGE = (0.25, 1.1)
EXPERIMENT_NAME = (
    f"{EXPERIMENT_BASE_NAME}__mode-{MIMIC_MODE}"
    f"__capacity-{CAPACITY:g}__neighbors-{N_NEIGHBORS}"
    f"__lambda-{LAMBDA_RANGE[0]:g}-{LAMBDA_RANGE[1]:g}"
    f"__rows-{DATASET_N_ROWS}"
)

# Optional overrides; leave as None to use the profile and default artifact directory.
CONFIG_PATH = None
ARTIFACT_DIR = None


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "MIMIC" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENT_ROOT = PROJECT_ROOT / "manuscript" / "experiments"
for path in [PROJECT_ROOT / "src", EXPERIMENT_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from streamlined.config import (
    experiment_artifact_dir,
    ensure_artifact_dirs,
    load_config,
    with_dataset_n_rows,
)
from streamlined.datasets import dataset_registry
from streamlined.runner import artifact_manifest, run_profile


In [ ]:
config_path = Path(CONFIG_PATH) if CONFIG_PATH else EXPERIMENT_ROOT / "configs" / "full.yaml"
artifact_base_dir = ARTIFACT_DIR or EXPERIMENT_ROOT / "artifacts"
artifact_dir = experiment_artifact_dir(artifact_base_dir, EXPERIMENT_NAME)
config = load_config(config_path, artifact_dir=artifact_dir)
config = with_dataset_n_rows(config, DATASET_N_ROWS)
config = replace(
    config,
    mimic_mode=MIMIC_MODE,
    mimic_capacity=CAPACITY,
    n_neighbors=N_NEIGHBORS,
    lambda_range=LAMBDA_RANGE,
)
ensure_artifact_dirs(config)

print(f"Profile: {config.profile.name}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Artifact directory: {config.artifact_root}")
print(f"Restart from scratch: {RESTART}")
print(f"Rows per dataset: {DATASET_N_ROWS:,}")
print(
    f"Mode: {config.mimic_mode}; capacity: {config.mimic_capacity}; "
    f"neighbors: {config.n_neighbors}; "
    f"lambda range: {config.lambda_range}"
)
display(dataset_registry())
display(artifact_manifest(config)[["artifact", "relative_path", "exists"]])


In [ ]:
tables = run_profile(
    config,
    run_experiment=True,
    show_progress=SHOW_PROGRESS,
    restart=RESTART,
)

print(f"Completed raw conditions: {len(tables['raw_results']):,}")
display(artifact_manifest(config)[["artifact", "relative_path", "exists"]])
display(tables["raw_results"].tail())
